# 01. 한국 의약품 ID 온톨로지 정리

모든 한국 데이터셋을 연결하는 식별자 체계를 정리한다.

| 식별자 | 설명 | 주로 쓰는 데이터셋 |
|---|---|---|
| **품목일련번호** | 식약처 허가 단위 기본 키 | 낱알식별, e약은요, DUR |
| **표준코드 (바코드)** | 유통 단위 코드 | 처방전, 약국 POS |
| **성분코드** | 주성분 단위 코드 | DUR 병용금기 |
| **ATC 코드** | WHO 분류 체계 | DrugBank, 국제 매핑 |

> **핵심 목표**: `품목일련번호` ↔ `성분코드` ↔ `ATC` 간 crosswalk 테이블 생성

In [5]:
import os, json
import requests
import pandas as pd
from pathlib import Path
from urllib.parse import quote

ROOT = Path("../..").resolve()
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass

DATAGOKR_KEY = os.environ.get("DATAGOKR_NADAL_KEY", "")

In [6]:
# 낱알식별 API — 첫 100건 조회하여 스키마 확인
def fetch_nadal(page=1, rows=100):
    url = "http://apis.data.go.kr/1471000/MdcinGrnIdntfcInfoService03/getMdcinGrnIdntfcInfoList03"
    params = {
        "serviceKey": DATAGOKR_KEY,
        "pageNo": page,
        "numOfRows": rows,
        "type": "json"
    }
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    return resp.json()

sample = fetch_nadal()
print(json.dumps(sample, indent=2, ensure_ascii=False)[:2000])

{
  "header": {
    "resultCode": "00",
    "resultMsg": "NORMAL SERVICE."
  },
  "body": {
    "pageNo": 1,
    "totalCount": 25518,
    "numOfRows": 100,
    "items": [
      {
        "ITEM_SEQ": "200808876",
        "ITEM_NAME": "가스디알정50밀리그램(디메크로틴산마그네슘)",
        "ENTP_SEQ": "19540006",
        "ENTP_NAME": "일동제약(주)",
        "CHART": "녹색의 원형 필름코팅정",
        "ITEM_IMAGE": "https://nedrug.mfds.go.kr/pbp/cmn/itemImageDownload/147426403087300104",
        "PRINT_FRONT": "IDG",
        "PRINT_BACK": null,
        "DRUG_SHAPE": "원형",
        "COLOR_CLASS1": "연두",
        "COLOR_CLASS2": null,
        "LINE_FRONT": null,
        "LINE_BACK": null,
        "LENG_LONG": "7.6",
        "LENG_SHORT": "7.6",
        "THICK": "3.6",
        "IMG_REGIST_TS": "20100326",
        "CLASS_NO": "02390",
        "CLASS_NAME": "기타의 소화기관용약",
        "ETC_OTC_NAME": "전문의약품",
        "ITEM_PERMIT_DATE": "20080820",
        "FORM_CODE_NAME": "당의정",
        "MARK_CODE_FRONT_ANAL": "",
        "MARK_CODE_BA

In [7]:
# 스키마 및 컬럼 확인
items = sample.get("body", {}).get("items", [])
if items:
    df_sample = pd.DataFrame(items)
    print("컬럼:", df_sample.columns.tolist())
    print("\n샘플 1행:")
    display(df_sample.head(1).T)

컬럼: ['ITEM_SEQ', 'ITEM_NAME', 'ENTP_SEQ', 'ENTP_NAME', 'CHART', 'ITEM_IMAGE', 'PRINT_FRONT', 'PRINT_BACK', 'DRUG_SHAPE', 'COLOR_CLASS1', 'COLOR_CLASS2', 'LINE_FRONT', 'LINE_BACK', 'LENG_LONG', 'LENG_SHORT', 'THICK', 'IMG_REGIST_TS', 'CLASS_NO', 'CLASS_NAME', 'ETC_OTC_NAME', 'ITEM_PERMIT_DATE', 'FORM_CODE_NAME', 'MARK_CODE_FRONT_ANAL', 'MARK_CODE_BACK_ANAL', 'MARK_CODE_FRONT_IMG', 'MARK_CODE_BACK_IMG', 'ITEM_ENG_NAME', 'CHANGE_DATE', 'MARK_CODE_FRONT', 'MARK_CODE_BACK', 'EDI_CODE', 'BIZRNO', 'STD_CD']

샘플 1행:


,0
ITEM_SEQ,200808876
ITEM_NAME,가스디알정50밀리그램(디메크로틴산마그네슘)
ENTP_SEQ,19540006
ENTP_NAME,일동제약(주)
CHART,녹색의 원형 필름코팅정
ITEM_IMAGE,https://nedrug.mfds.go.kr/pbp/cmn/itemImageDow...
PRINT_FRONT,IDG
PRINT_BACK,None
DRUG_SHAPE,원형
COLOR_CLASS1,연두


In [8]:
# 핵심 식별자 컬럼 존재 여부 확인
id_cols = ["ITEM_SEQ", "ITEM_NAME", "ENTP_NAME", "CHART", "DRUG_SHAPE", "COLOR_CLASS1",
           "PRINT_FRONT", "PRINT_BACK", "ITEM_IMAGE"]

if items:
    found = [c for c in id_cols if c in df_sample.columns]
    missing = [c for c in id_cols if c not in df_sample.columns]
    print("존재:", found)
    print("누락:", missing)

존재: ['ITEM_SEQ', 'ITEM_NAME', 'ENTP_NAME', 'CHART', 'DRUG_SHAPE', 'COLOR_CLASS1', 'PRINT_FRONT', 'PRINT_BACK', 'ITEM_IMAGE']
누락: []
